# Backend Mini Simulator

Notebook ini menjalankan workflow simulator lewat backend asli: `ProjectConfig`, grid builder, transmissibility, initializer, validation, runner, results, residual, dan Jacobian. Tidak ada dependency ke UI PySide.

## 1. Setup import backend

Jalankan notebook dari root repo. Kalau notebook dibuka dari lokasi lain, cell ini tetap mencoba mencari folder repo yang punya `engine/` dan `modules/`.

In [ ]:
from pathlib import Path
import sys
from html import escape

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'engine').exists() and (candidate / 'modules').exists():
            return candidate
    fallback = Path(r'C:\My Workspaces\venv.Python\SIMULASI RESERVOIR')
    if (fallback / 'engine').exists() and (fallback / 'modules').exists():
        return fallback
    raise RuntimeError('Repo root tidak ditemukan. Jalankan notebook dari folder project.')

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from modules.project_service import (
    create_empty_project,
    update_grid_spec,
    update_grid_connectivity,
    update_initial_conditions,
    update_solver_config,
    update_wells,
    update_perturbation_config,
    update_method_config,
)
from modules.validation_service import validate_project
from modules.simulation_service import run_simulation
from modules.results_service import get_run_summary, get_latest_step, get_all_cell_properties
from engine.domain.project import MethodConfig, PerturbationConfig, WellConfig
from engine.grid.builder import build_grid
from engine.physics.transmissibility import update_grid_transmissibility
from engine.simulation.initializer import initialize_state

try:
    from IPython.display import HTML, display
except Exception:
    HTML = None
    display = print

print(f'Repo root: {ROOT}')

In [ ]:
def fmt(value):
    if isinstance(value, float):
        if abs(value) >= 1e4 or (0 < abs(value) < 1e-3):
            return f'{value:.4e}'
        return f'{value:.6g}'
    return str(value)

def show_table(rows, columns=None, title=None, max_rows=30):
    rows = list(rows)
    if not rows:
        print(f'{title + ": " if title else ""}no rows')
        return
    if columns is None:
        columns = list(rows[0].keys())
    visible_rows = rows[:max_rows]
    if HTML is None:
        if title:
            print(title)
        for row in visible_rows:
            print({col: fmt(row.get(col, '')) for col in columns})
        return
    html = ''
    if title:
        html += f'<h4 style="font-family:Segoe UI;margin:10px 0 6px;color:#1F2937">{escape(title)}</h4>'
    html += '<table style="border-collapse:collapse;font-family:Segoe UI;font-size:13px">'
    html += '<thead><tr>' + ''.join(
        f'<th style="background:#F1F4F8;color:#0F5C8E;border:1px solid #D7DEE7;padding:6px 10px;text-align:left">{escape(col)}</th>'
        for col in columns
    ) + '</tr></thead><tbody>'
    for row in visible_rows:
        html += '<tr>' + ''.join(
            f'<td style="border:1px solid #D7DEE7;padding:5px 10px;color:#1F2937">{escape(fmt(row.get(col, "")))}</td>'
            for col in columns
        ) + '</tr>'
    html += '</tbody></table>'
    if len(rows) > max_rows:
        html += f'<div style="font-family:Segoe UI;color:#5B6676;font-size:12px;margin-top:4px">Showing {max_rows} of {len(rows)} rows.</div>'
    display(HTML(html))

## 2. Buat project case kecil

Case default sengaja kecil: 2 x 2 x 1 cells, satu production well kecil. Tujuannya supaya notebook cepat, tetap menghasilkan residual dan Jacobian, dan tetap konvergen.

In [ ]:
project = create_empty_project('Notebook Mini Backend Simulator')

update_grid_spec(project, nx=2, ny=2, nz=1, dx=100.0, dy=100.0, dz=50.0)
update_grid_connectivity(project, 5)
update_initial_conditions(
    project,
    reference_depth=6500.0,
    reference_pressure=2500.0,
    initial_sw=0.25,
    initial_sg=0.05,
)
update_solver_config(
    project,
    initial_timestep_days=0.01,
    min_timestep_days=0.0025,
    max_time_days=0.01,
    max_newton_iterations=12,
    residual_tolerance=5e-2,
    max_pressure_correction=10.0,
    max_saturation_correction=0.001,
)
update_wells(
    project,
    [WellConfig(name='P1', well_type='production', cell_id=1, well_model='simple_flowrate', flowrate=0.01)],
)
update_perturbation_config(project, PerturbationConfig(perturbed_cell_id=1))
update_method_config(project, MethodConfig(active_method='newton_raphson'))

case_rows = [
    {'item': 'grid', 'value': f'{project.grid_spec.nx} x {project.grid_spec.ny} x {project.grid_spec.nz}'},
    {'item': 'connectivity', 'value': project.grid_spec.connectivity},
    {'item': 'initial pressure', 'value': project.reference_data.reference_pressure},
    {'item': 'initial Sw', 'value': project.initial_conditions.initial_sw},
    {'item': 'initial Sg', 'value': project.initial_conditions.initial_sg},
    {'item': 'dt / max time', 'value': f'{project.solver.initial_timestep_days} / {project.solver.max_time_days} days'},
    {'item': 'well', 'value': f'{project.wells[0].name}, cell {project.wells[0].cell_id}, q={project.wells[0].flowrate}'},
]
show_table(case_rows, title='Project configuration')

## 3. Validasi run-readiness

Ini memakai `modules.validation_service.validate_project()`, sama seperti service run UI.

In [ ]:
errors = validate_project(project)
if errors:
    raise ValueError('Project belum valid:\n' + '\n'.join(f'- {err}' for err in errors))
print('Project valid. Semua constraint backend siap run.')

## 4. Inspect grid, transmissibility, dan initial state

Cell ini menjalankan bagian awal workflow engine: build grid, attach connections, update transmissibility, lalu initialize pressure/saturations.

In [ ]:
grid = build_grid(project)
update_grid_transmissibility(grid)
state0 = initialize_state(project, grid)

cell_rows = []
for cell in grid.cells:
    cell_rows.append({
        'cell': cell.cell_id + 1,
        'i': cell.i,
        'j': cell.j,
        'k': cell.k,
        'poro': cell.porosity,
        'kx': cell.perm_x,
        'p0': state0.pressure[cell.cell_id],
        'Sw0': state0.sw[cell.cell_id],
        'Sg0': state0.sg[cell.cell_id],
    })
show_table(cell_rows, title='Initial cells')

conn_rows = []
for c in grid.connections:
    conn_rows.append({
        'from': c.from_cell_id + 1,
        'to': c.to_cell_id + 1,
        'direction': c.direction,
        'area': c.area,
        'distance': c.distance,
        'transmissibility': c.transmissibility,
    })
show_table(conn_rows, title='Grid connections')

## 5. Run simulation

Ini memakai `modules.simulation_service.run_simulation()`. Callback iteration disimpan supaya kita bisa lihat history Newton.

In [ ]:
iteration_history = []

def on_iteration(time_days, iteration, residual_norm, converged):
    iteration_history.append({
        'time_days': time_days,
        'iteration': iteration,
        'residual_norm': residual_norm,
        'converged': converged,
    })

result = run_simulation(project, on_iteration=on_iteration)
latest = get_latest_step(result)
summary = get_run_summary(result)

show_table([summary], title='Run summary')
show_table(iteration_history, title='Newton iteration history')
print('warnings:', result.warnings)

## 6. Latest step: state, residual, dan evaluated properties

In [ ]:
if latest is None:
    raise RuntimeError('Run tidak menghasilkan timestep.')

state_rows = []
for i, (p, sw, sg, so) in enumerate(zip(latest.pressure, latest.sw, latest.sg, latest.so), start=1):
    state_rows.append({'cell': i, 'pressure': p, 'Sw': sw, 'Sg': sg, 'So': so})
show_table(state_rows, title='Final state per cell')

residual_rows = []
for i in range(len(latest.pressure)):
    residual_rows.append({
        'cell': i + 1,
        'Ro': latest.oil_residual_per_cell[i],
        'Rw': latest.water_residual_per_cell[i],
        'Rg': latest.gas_residual_per_cell[i],
        'R_total': latest.residual_per_cell[i],
    })
show_table(residual_rows, title='Residual per cell')

props = get_all_cell_properties(project, latest)
prop_cols = ['cell', 'pressure_psia', 'so', 'sw', 'sg', 'bo', 'bw', 'bg', 'kro', 'krw', 'krg', 'lam_o', 'lam_w', 'lam_g']
show_table(props, columns=prop_cols, title='Evaluated PVT/rock properties')

## 7. Jacobian layout dan diag cell 3x3

Backend menyusun unknown/residual global sebagai `[p semua cell, Sw semua cell, Sg semua cell]` dan residual sebagai `[Ro semua cell, Rw semua cell, Rg semua cell]`. Jadi untuk cell ke-i:

- diag p = `J[i, i]` = dRo/dp
- diag Sw = `J[N+i, N+i]` = dRw/dSw
- diag Sg = `J[2N+i, 2N+i]` = dRg/dSg

In [ ]:
def jacobian_cell_count(jacobian):
    size = len(jacobian)
    if size == 0 or size % 3 != 0:
        return 0
    return size // 3

def cell_jacobian_block(jacobian, cell_id):
    n = jacobian_cell_count(jacobian)
    if n <= 0:
        raise ValueError('Jacobian kosong atau bukan 3N x 3N.')
    i = cell_id - 1
    rows = [i, n + i, 2 * n + i]
    cols = [i, n + i, 2 * n + i]
    row_labels = ['Ro', 'Rw', 'Rg']
    col_labels = ['p', 'Sw', 'Sg']
    return [
        {'residual': row_labels[r], **{col_labels[c]: jacobian[rows[r]][cols[c]] for c in range(3)}}
        for r in range(3)
    ]

def diag_cell_rows(jacobian):
    n = jacobian_cell_count(jacobian)
    rows = []
    for i in range(n):
        rows.append({
            'cell': i + 1,
            'diag_p_dRo_dp': jacobian[i][i],
            'diag_Sw_dRw_dSw': jacobian[n + i][n + i],
            'diag_Sg_dRg_dSg': jacobian[2 * n + i][2 * n + i],
        })
    return rows

J = latest.jacobian
print(f'Jacobian size: {len(J)} x {len(J[0]) if J else 0}')
show_table(cell_jacobian_block(J, cell_id=1), title='Cell 1 self-block Jacobian 3x3')
show_table(diag_cell_rows(J), title='Diag cell per cell')

## 8. Mini symmetry check untuk diag cell

Ini versi notebook dari konsep Simetri Jacobian: bandingkan diag p/Sw/Sg antara selected cell dan pasangan simetris secara geometri terhadap well cell.

In [ ]:
def symmetric_cells(cell_id, well_cell, nx, ny):
    nr, nc = divmod(cell_id - 1, nx)
    wr, wc = divmod(well_cell - 1, nx)
    dr, dc = nr - wr, nc - wc
    result = set()
    for tr, tc in [(dr, dc), (-dr, dc), (dr, -dc), (-dr, -dc), (dc, dr), (-dc, dr), (dc, -dr), (-dc, -dr)]:
        r2, c2 = wr + tr, wc + tc
        if 0 <= r2 < ny and 0 <= c2 < nx:
            mapped = r2 * nx + c2 + 1
            if mapped != cell_id:
                result.add(mapped)
    return sorted(result)

def rel_diff(a, b):
    return abs(a - b) / max(abs(a), abs(b), 1e-30)

def diag_symmetry_report(jacobian, selected_cell, well_cell, nx, ny):
    diag_rows = {row['cell']: row for row in diag_cell_rows(jacobian)}
    rows = []
    for sym_cell in symmetric_cells(selected_cell, well_cell, nx, ny):
        a = diag_rows[selected_cell]
        b = diag_rows[sym_cell]
        rows.append({
            'selected': selected_cell,
            'symmetric': sym_cell,
            'p_rel': rel_diff(a['diag_p_dRo_dp'], b['diag_p_dRo_dp']),
            'Sw_rel': rel_diff(a['diag_Sw_dRw_dSw'], b['diag_Sw_dRw_dSw']),
            'Sg_rel': rel_diff(a['diag_Sg_dRg_dSg'], b['diag_Sg_dRg_dSg']),
            'max_rel': max(
                rel_diff(a['diag_p_dRo_dp'], b['diag_p_dRo_dp']),
                rel_diff(a['diag_Sw_dRw_dSw'], b['diag_Sw_dRw_dSw']),
                rel_diff(a['diag_Sg_dRg_dSg'], b['diag_Sg_dRg_dSg']),
            ),
        })
    return rows

selected_cell = 2
well_cell = project.wells[0].cell_id
sym_rows = diag_symmetry_report(J, selected_cell, well_cell, project.grid_spec.nx, project.grid_spec.ny)
show_table(sym_rows, title=f'Diag-cell symmetry around well cell {well_cell}')

## 9. Optional quick plots

Plot ini opsional. Kalau `matplotlib` tidak tersedia, cell akan skip tanpa mengganggu hasil backend.

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot([r['iteration'] for r in iteration_history], [r['residual_norm'] for r in iteration_history], marker='o')
    axes[0].set_title('Newton residual norm')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Residual norm')
    axes[0].grid(True, alpha=0.3)

    cells = [row['cell'] for row in state_rows]
    axes[1].plot(cells, [row['pressure'] for row in state_rows], marker='o', label='Pressure')
    axes[1].set_title('Final pressure')
    axes[1].set_xlabel('Cell')
    axes[1].set_ylabel('psia')
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f'Plot skipped: {exc}')